In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\n🔍 GPU Check:")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✅ GPU ready!")
else:
    print("\n⚠️  GPU not detected! Go to: Runtime → Change runtime type → GPU")

In [ ]:
# Upload dataset
from google.colab import files
import os

print('📦 Upload dataset.zip (345MB)')
if os.path.exists('dataset.zip'):
    print('✅ dataset.zip exists')
    response = input('Re-upload? (y/n): ')
    if response.lower() == 'y':
        uploaded = files.upload()
else:
    uploaded = files.upload()
print('✅ Upload complete!')

In [ ]:
# Extract dataset
if not os.path.exists('dataset'):
    print('📂 Extracting...')
    !unzip -q dataset.zip
    print('✅ Complete!')
else:
    print('✅ Dataset ready!')

# Count samples
train_bot = len(os.listdir('dataset/train/bot'))
train_not_bot = len(os.listdir('dataset/train/not_bot'))
val_bot = len(os.listdir('dataset/val/bot'))
val_not_bot = len(os.listdir('dataset/val/not_bot'))

print(f'\n📊 Dataset:')
print(f'   Train: {train_bot} bots, {train_not_bot} not_bots')
print(f'   Val: {val_bot} bots, {val_not_bot} not_bots')
print(f'   Balance: {train_bot/(train_bot + train_not_bot)*100:.1f}% bots')

In [ ]:
# Import libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import numpy as np
import cv2
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Libraries loaded! Device: {DEVICE}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# Configuration
SELECTED_CONFIG = 0  # 0-5 or 'all'

CONFIGS = [
    {'name': 'mobilenet_equal', 'model': 'mobilenet_v2', 'class_weight_bot': 1.0, 'batch_size': 64, 'lr': 0.001, 'epochs': 15},
    {'name': 'mobilenet_weighted', 'model': 'mobilenet_v2', 'class_weight_bot': 1.65, 'batch_size': 64, 'lr': 0.001, 'epochs': 15},
    {'name': 'resnet18_equal', 'model': 'resnet18', 'class_weight_bot': 1.0, 'batch_size': 64, 'lr': 0.001, 'epochs': 15},
    {'name': 'resnet18_weighted', 'model': 'resnet18', 'class_weight_bot': 1.65, 'batch_size': 64, 'lr': 0.001, 'epochs': 15},
    {'name': 'resnet34_equal', 'model': 'resnet34', 'class_weight_bot': 1.0, 'batch_size': 64, 'lr': 0.001, 'epochs': 15},
    {'name': 'resnet34_weighted', 'model': 'resnet34', 'class_weight_bot': 1.65, 'batch_size': 64, 'lr': 0.001, 'epochs': 15},
]

configs_to_run = CONFIGS if SELECTED_CONFIG == 'all' else [CONFIGS[SELECTED_CONFIG]]
print(f"🎯 Running {len(configs_to_run)} config(s):")
for cfg in configs_to_run:
    print(f"  - {cfg['name']}")

all_results = []

In [ ]:
# Dataset
class AvatarDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        for label_dir in ['bot', 'not_bot']:
            label = 0 if label_dir == 'bot' else 1
            for img_path in (self.root_dir / label_dir).glob('*.png'):
                self.samples.append((str(img_path), label))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        return self.transform(image) if self.transform else image, label

train_transform = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = AvatarDataset('dataset/train', train_transform)
test_dataset = AvatarDataset('dataset/val', test_transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
print(f'✅ Data loaders ready')

In [ ]:
# Training functions
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return loss_sum / len(loader), 100. * correct / total

def test_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss_sum += criterion(outputs, labels).item()
            probs = torch.softmax(outputs, 1)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return (loss_sum / len(loader), 100. * correct / total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))

print('✅ Training functions defined')

In [ ]:
# Train models
for config_idx, config in enumerate(configs_to_run):
    print("\n" + "="*80)
    print(f"TRAINING: {config['name']} ({config_idx+1}/{len(configs_to_run)})")
    print("="*80)
    
    # Build model
    if config['model'] == 'mobilenet_v2':
        model = torchvision.models.mobilenet_v2(pretrained=True)
        model.classifier[1] = nn.Linear(model.last_channel, 2)
    elif config['model'] == 'resnet18':
        model = torchvision.models.resnet18(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif config['model'] == 'resnet34':
        model = torchvision.models.resnet34(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, 2)
    
    model = model.to(DEVICE)
    
    class_weights = torch.tensor([config['class_weight_bot'], 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=2, factor=0.5)
    
    best_f1 = 0.0
    best_recall = 0.0
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'bot_recall': [], 'bot_precision': [], 'bot_f1': []}
    
    start_time = time.time()
    
    # Training loop
    print("🏋️ Training...")
    for epoch in range(config['epochs']):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        test_loss, test_acc, preds, labels, probs = test_epoch(model, test_loader, criterion, DEVICE)
        
        cm = confusion_matrix(labels, preds)
        bot_recall = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0
        bot_precision = cm[0,0] / (cm[0,0] + cm[1,0]) if (cm[0,0] + cm[1,0]) > 0 else 0
        bot_f1 = 2 * bot_precision * bot_recall / (bot_precision + bot_recall) if (bot_precision + bot_recall) > 0 else 0
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['bot_recall'].append(bot_recall)
        history['bot_precision'].append(bot_precision)
        history['bot_f1'].append(bot_f1)
        
        print(f'Epoch {epoch+1:2d}/{config["epochs"]} | Train: {train_acc:5.2f}% | Test: {test_acc:5.2f}% | Recall: {bot_recall:.4f} | F1: {bot_f1:.4f}', end='')
        
        scheduler.step(bot_f1)
        
        # Save best model: prioritize recall=1.0, then F1
        is_better = False
        if bot_recall >= 0.999:  # Near 100% recall
            if best_recall < 0.999 or bot_f1 > best_f1:
                is_better = True
        elif bot_recall > best_recall:
            is_better = True
        
        if is_better:
            best_recall = bot_recall
            best_f1 = bot_f1
            model_filename = f"{config['name']}.pth"
            torch.save(model.state_dict(), model_filename)
            print(' ✓ [BEST]')
        else:
            print()
    
    training_time = time.time() - start_time
    
    # Final evaluation
    model.load_state_dict(torch.load(model_filename))
    test_loss, test_acc, preds, labels, probs = test_epoch(model, test_loader, criterion, DEVICE)
    cm = confusion_matrix(labels, preds)
    bot_recall = cm[0,0] / (cm[0,0] + cm[0,1])
    bot_precision = cm[0,0] / (cm[0,0] + cm[1,0])
    bot_f1 = 2 * bot_precision * bot_recall / (bot_precision + bot_recall)
    roc_auc = roc_auc_score(labels, probs[:, 0])
    
    print(f"\n✅ RESULTS:")
    print(f"   Accuracy: {test_acc:.2f}%")
    print(f"   Bot Recall: {bot_recall:.4f}")
    print(f"   Bot Precision: {bot_precision:.4f}")
    print(f"   Bot F1: {bot_f1:.4f}")
    print(f"   ROC AUC: {roc_auc:.4f}")
    print(f"   Time: {training_time/60:.1f} min")
    
    # Store results
    all_results.append({
        'config_name': config['name'],
        'model': config['model'],
        'class_weight_bot': config['class_weight_bot'],
        'test_accuracy': test_acc,
        'bot_recall': bot_recall,
        'bot_precision': bot_precision,
        'bot_f1': bot_f1,
        'roc_auc': roc_auc,
        'training_time': training_time,
        'confusion_matrix': cm,
        'history': history,
        'predictions': preds,
        'labels': labels,
        'probabilities': probs,
    })

print(f"\n{'='*80}")
print(f"✅ TRAINING COMPLETE!")
print(f"{'='*80}")

In [ ]:
# Select best model: highest F1 with recall >= 0.999
models_with_perfect_recall = [r for r in all_results if r['bot_recall'] >= 0.999]

if models_with_perfect_recall:
    best_result = max(models_with_perfect_recall, key=lambda r: r['bot_f1'])
    selection_criteria = "Highest F1 with 100% recall"
else:
    best_result = max(all_results, key=lambda r: r['bot_recall'])
    selection_criteria = "Highest recall"

print("\n" + "="*80)
print(f"🏆 BEST MODEL: {best_result['config_name']}")
print(f"Selection: {selection_criteria}")
print("="*80)
print(f"Bot Recall: {best_result['bot_recall']:.4f}")
print(f"Bot Precision: {best_result['bot_precision']:.4f}")
print(f"Bot F1: {best_result['bot_f1']:.4f}")
print(f"Test Accuracy: {best_result['test_accuracy']:.2f}%")
print(f"ROC AUC: {best_result['roc_auc']:.4f}")
print("="*80)

if len(all_results) > 1:
    df = pd.DataFrame([{
        'Config': r['config_name'],
        'Recall': f"{r['bot_recall']:.4f}",
        'Precision': f"{r['bot_precision']:.4f}",
        'F1': f"{r['bot_f1']:.4f}",
        'Acc': f"{r['test_accuracy']:.2f}%",
    } for r in all_results])
    print("\n📊 All Models:")
    print(df.to_string(index=False))

In [ ]:
# Advanced visualizations with dynamic scaling
if len(all_results) > 1:
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    config_names = [r['config_name'] for r in all_results]
    recalls = [r['bot_recall'] for r in all_results]
    precisions = [r['bot_precision'] for r in all_results]
    f1s = [r['bot_f1'] for r in all_results]
    
    # Color by model
    colors = ['blue' if 'mobilenet' in n else 'red' if 'resnet34' in n else 'green' for n in config_names]
    
    # 1. Bot Recall (dynamic scale)
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.bar(range(len(config_names)), recalls, color=colors, alpha=0.7)
    ax1.set_xticks(range(len(config_names)))
    ax1.set_xticklabels(config_names, rotation=45, ha='right', fontsize=8)
    ax1.set_ylabel('Bot Recall')
    ax1.set_title('Bot Recall Comparison')
    # Dynamic y-axis
    ymin, ymax = min(recalls), max(recalls)
    margin = (ymax - ymin) * 0.1 if ymax > ymin else 0.05
    ax1.set_ylim(max(0, ymin - margin), min(1, ymax + margin))
    ax1.grid(True, axis='y', alpha=0.3)
    ax1.axhline(y=best_result['bot_recall'], color='r', linestyle='--', alpha=0.5)
    
    # 2. Bot F1 (dynamic scale)
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.bar(range(len(config_names)), f1s, color=colors, alpha=0.7)
    ax2.set_xticks(range(len(config_names)))
    ax2.set_xticklabels(config_names, rotation=45, ha='right', fontsize=8)
    ax2.set_ylabel('Bot F1 Score')
    ax2.set_title('Bot F1 Score Comparison')
    ymin, ymax = min(f1s), max(f1s)
    margin = (ymax - ymin) * 0.1 if ymax > ymin else 0.05
    ax2.set_ylim(max(0, ymin - margin), min(1, ymax + margin))
    ax2.grid(True, axis='y', alpha=0.3)
    
    # 3. Recall vs Precision scatter (dynamic scale)
    ax3 = fig.add_subplot(gs[0, 2])
    for i, name in enumerate(config_names):
        ax3.scatter(recalls[i], precisions[i], s=200, alpha=0.7, color=colors[i], label=name)
    ax3.set_xlabel('Bot Recall')
    ax3.set_ylabel('Bot Precision')
    ax3.set_title('Recall vs Precision')
    # Dynamic axes
    rec_margin = (max(recalls) - min(recalls)) * 0.1 if max(recalls) > min(recalls) else 0.05
    prec_margin = (max(precisions) - min(precisions)) * 0.1 if max(precisions) > min(precisions) else 0.05
    ax3.set_xlim(max(0, min(recalls) - rec_margin), min(1, max(recalls) + rec_margin))
    ax3.set_ylim(max(0, min(precisions) - prec_margin), min(1, max(precisions) + prec_margin))
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax3.grid(True, alpha=0.3)
    
    # 4. Confusion matrix heatmap (best model)
    ax4 = fig.add_subplot(gs[1, 0])
    cm_best = best_result['confusion_matrix']
    sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=ax4, cbar=True)
    ax4.set_title(f'Confusion Matrix: {best_result["config_name"]}')
    ax4.set_xlabel('Predicted')
    ax4.set_ylabel('Actual')
    ax4.set_xticklabels(['Bot', 'Not Bot'])
    ax4.set_yticklabels(['Bot', 'Not Bot'])
    
    # 5. ROC curves
    ax5 = fig.add_subplot(gs[1, 1])
    for i, result in enumerate(all_results):
        fpr, tpr, _ = roc_curve(result['labels'], result['probabilities'][:, 0])
        ax5.plot(fpr, tpr, label=f"{result['config_name']} (AUC={result['roc_auc']:.3f})", linewidth=2)
    ax5.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
    ax5.set_xlabel('False Positive Rate')
    ax5.set_ylabel('True Positive Rate')
    ax5.set_title('ROC Curves')
    ax5.legend(fontsize=7)
    ax5.grid(True, alpha=0.3)
    
    # 6. Precision-Recall curves
    ax6 = fig.add_subplot(gs[1, 2])
    for i, result in enumerate(all_results):
        precision, recall, _ = precision_recall_curve(result['labels'], result['probabilities'][:, 0], pos_label=0)
        ax6.plot(recall, precision, label=result['config_name'], linewidth=2)
    ax6.set_xlabel('Recall')
    ax6.set_ylabel('Precision')
    ax6.set_title('Precision-Recall Curves')
    ax6.legend(fontsize=7)
    ax6.grid(True, alpha=0.3)
    
    # 7-9. Training curves (best model)
    history = best_result['history']
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax7 = fig.add_subplot(gs[2, 0])
    ax7.plot(epochs, history['train_loss'], 'o-', label='Train', linewidth=2)
    ax7.plot(epochs, history['test_loss'], 's-', label='Test', linewidth=2)
    ax7.set_xlabel('Epoch')
    ax7.set_ylabel('Loss')
    ax7.set_title(f'Loss: {best_result["config_name"]}')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    
    ax8 = fig.add_subplot(gs[2, 1])
    ax8.plot(epochs, history['train_acc'], 'o-', label='Train', linewidth=2)
    ax8.plot(epochs, history['test_acc'], 's-', label='Test', linewidth=2)
    ax8.set_xlabel('Epoch')
    ax8.set_ylabel('Accuracy (%)')
    ax8.set_title(f'Accuracy: {best_result["config_name"]}')
    # Dynamic scale
    all_acc = history['train_acc'] + history['test_acc']
    acc_min, acc_max = min(all_acc), max(all_acc)
    margin = (acc_max - acc_min) * 0.1 if acc_max > acc_min else 5
    ax8.set_ylim(max(0, acc_min - margin), min(100, acc_max + margin))
    ax8.legend()
    ax8.grid(True, alpha=0.3)
    
    ax9 = fig.add_subplot(gs[2, 2])
    ax9.plot(epochs, history['bot_recall'], 'o-', color='green', label='Recall', linewidth=2)
    ax9.plot(epochs, history['bot_precision'], 's-', color='blue', label='Precision', linewidth=2)
    ax9.plot(epochs, history['bot_f1'], '^-', color='red', label='F1', linewidth=2)
    ax9.set_xlabel('Epoch')
    ax9.set_ylabel('Score')
    ax9.set_title(f'Bot Metrics: {best_result["config_name"]}')
    # Dynamic scale
    all_metrics = history['bot_recall'] + history['bot_precision'] + history['bot_f1']
    met_min, met_max = min(all_metrics), max(all_metrics)
    margin = (met_max - met_min) * 0.1 if met_max > met_min else 0.05
    ax9.set_ylim(max(0, met_min - margin), min(1, met_max + margin))
    ax9.legend()
    ax9.grid(True, alpha=0.3)
    
    plt.savefig('comprehensive_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Saved: comprehensive_analysis.png")
    
    # Save individual plots
    print("\n📸 Saving individual plots...")
    
    # 1. Bot Recall
    fig1, ax = plt.subplots(figsize=(10, 6))
    ax.bar(range(len(config_names)), recalls, color=colors, alpha=0.7)
    ax.set_xticks(range(len(config_names)))
    ax.set_xticklabels(config_names, rotation=45, ha='right')
    ax.set_ylabel('Bot Recall')
    ax.set_title('Bot Recall Comparison')
    ymin, ymax = min(recalls), max(recalls)
    margin = (ymax - ymin) * 0.1 if ymax > ymin else 0.05
    ax.set_ylim(max(0, ymin - margin), min(1, ymax + margin))
    ax.grid(True, axis='y', alpha=0.3)
    ax.axhline(y=best_result['bot_recall'], color='r', linestyle='--', alpha=0.5, label='Best')
    ax.legend()
    plt.tight_layout()
    plt.savefig('01_bot_recall_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 01_bot_recall_comparison.png")
    
    # 2. Bot F1
    fig2, ax = plt.subplots(figsize=(10, 6))
    ax.bar(range(len(config_names)), f1s, color=colors, alpha=0.7)
    ax.set_xticks(range(len(config_names)))
    ax.set_xticklabels(config_names, rotation=45, ha='right')
    ax.set_ylabel('Bot F1 Score')
    ax.set_title('Bot F1 Score Comparison')
    ymin, ymax = min(f1s), max(f1s)
    margin = (ymax - ymin) * 0.1 if ymax > ymin else 0.05
    ax.set_ylim(max(0, ymin - margin), min(1, ymax + margin))
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('02_bot_f1_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 02_bot_f1_comparison.png")
    
    # 3. Recall vs Precision
    fig3, ax = plt.subplots(figsize=(10, 8))
    for i, name in enumerate(config_names):
        ax.scatter(recalls[i], precisions[i], s=300, alpha=0.7, color=colors[i], label=name)
    ax.set_xlabel('Bot Recall')
    ax.set_ylabel('Bot Precision')
    ax.set_title('Recall vs Precision Trade-off')
    rec_margin = (max(recalls) - min(recalls)) * 0.1 if max(recalls) > min(recalls) else 0.05
    prec_margin = (max(precisions) - min(precisions)) * 0.1 if max(precisions) > min(precisions) else 0.05
    ax.set_xlim(max(0, min(recalls) - rec_margin), min(1, max(recalls) + rec_margin))
    ax.set_ylim(max(0, min(precisions) - prec_margin), min(1, max(precisions) + prec_margin))
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('03_recall_vs_precision.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 03_recall_vs_precision.png")
    
    # 4. Confusion Matrix
    fig4, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(best_result['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=ax, cbar=True, annot_kws={'size': 16})
    ax.set_title(f'Confusion Matrix: {best_result["config_name"]}', fontsize=14)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_xticklabels(['Bot', 'Not Bot'])
    ax.set_yticklabels(['Bot', 'Not Bot'])
    plt.tight_layout()
    plt.savefig('04_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 04_confusion_matrix.png")
    
    # 5. ROC Curves
    fig5, ax = plt.subplots(figsize=(10, 8))
    for i, result in enumerate(all_results):
        fpr, tpr, _ = roc_curve(result['labels'], result['probabilities'][:, 0])
        ax.plot(fpr, tpr, label=f"{result['config_name']} (AUC={result['roc_auc']:.3f})", linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random', linewidth=2)
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves - All Models', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('05_roc_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 05_roc_curves.png")
    
    # 6. Precision-Recall Curves
    fig6, ax = plt.subplots(figsize=(10, 8))
    for i, result in enumerate(all_results):
        precision, recall, _ = precision_recall_curve(result['labels'], result['probabilities'][:, 0], pos_label=0)
        ax.plot(recall, precision, label=result['config_name'], linewidth=2)
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Curves - All Models', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('06_precision_recall_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 06_precision_recall_curves.png")
    
    # 7. Loss
    history = best_result['history']
    epochs = range(1, len(history['train_loss']) + 1)
    fig7, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['train_loss'], 'o-', label='Train', linewidth=2, markersize=8)
    ax.plot(epochs, history['test_loss'], 's-', label='Test', linewidth=2, markersize=8)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'Training Loss: {best_result["config_name"]}', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('07_training_loss.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 07_training_loss.png")
    
    # 8. Accuracy
    fig8, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['train_acc'], 'o-', label='Train', linewidth=2, markersize=8)
    ax.plot(epochs, history['test_acc'], 's-', label='Test', linewidth=2, markersize=8)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title(f'Training Accuracy: {best_result["config_name"]}', fontsize=14)
    all_acc = history['train_acc'] + history['test_acc']
    acc_min, acc_max = min(all_acc), max(all_acc)
    margin = (acc_max - acc_min) * 0.1 if acc_max > acc_min else 5
    ax.set_ylim(max(0, acc_min - margin), min(100, acc_max + margin))
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('08_training_accuracy.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 08_training_accuracy.png")
    
    # 9. Bot Metrics
    fig9, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, history['bot_recall'], 'o-', color='green', label='Recall', linewidth=2, markersize=8)
    ax.plot(epochs, history['bot_precision'], 's-', color='blue', label='Precision', linewidth=2, markersize=8)
    ax.plot(epochs, history['bot_f1'], '^-', color='red', label='F1', linewidth=2, markersize=8)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'Bot Detection Metrics: {best_result["config_name"]}', fontsize=14)
    all_metrics = history['bot_recall'] + history['bot_precision'] + history['bot_f1']
    met_min, met_max = min(all_metrics), max(all_metrics)
    margin = (met_max - met_min) * 0.1 if met_max > met_min else 0.05
    ax.set_ylim(max(0, met_min - margin), min(1, met_max + margin))
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('09_bot_metrics.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✓ 09_bot_metrics.png")
    
    print("\n✅ Saved 10 images total:")
    print("   • comprehensive_analysis.png (all 9 plots)")
    print("   • 01-09 individual plots")

else:
    # Single model visualizations
    result = all_results[0]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    history = result['history']
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0,0].plot(epochs, history['train_loss'], 'o-', label='Train')
    axes[0,0].plot(epochs, history['test_loss'], 's-', label='Test')
    axes[0,0].set_title('Loss')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # Accuracy (dynamic scale)
    axes[0,1].plot(epochs, history['train_acc'], 'o-', label='Train')
    axes[0,1].plot(epochs, history['test_acc'], 's-', label='Test')
    all_acc = history['train_acc'] + history['test_acc']
    acc_min, acc_max = min(all_acc), max(all_acc)
    margin = (acc_max - acc_min) * 0.1 if acc_max > acc_min else 5
    axes[0,1].set_ylim(max(0, acc_min - margin), min(100, acc_max + margin))
    axes[0,1].set_title('Accuracy')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Bot metrics (dynamic scale)
    axes[0,2].plot(epochs, history['bot_recall'], 'o-', label='Recall', color='green')
    axes[0,2].plot(epochs, history['bot_precision'], 's-', label='Precision', color='blue')
    axes[0,2].plot(epochs, history['bot_f1'], '^-', label='F1', color='red')
    all_metrics = history['bot_recall'] + history['bot_precision'] + history['bot_f1']
    met_min, met_max = min(all_metrics), max(all_metrics)
    margin = (met_max - met_min) * 0.1 if met_max > met_min else 0.05
    axes[0,2].set_ylim(max(0, met_min - margin), min(1, met_max + margin))
    axes[0,2].set_title('Bot Metrics')
    axes[0,2].legend()
    axes[0,2].grid(True, alpha=0.3)
    
    # Confusion matrix
    sns.heatmap(result['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[1,0])
    axes[1,0].set_title('Confusion Matrix')
    axes[1,0].set_xticklabels(['Bot', 'Not Bot'])
    axes[1,0].set_yticklabels(['Bot', 'Not Bot'])
    
    # ROC curve
    fpr, tpr, _ = roc_curve(result['labels'], result['probabilities'][:, 0])
    axes[1,1].plot(fpr, tpr, linewidth=2, label=f'AUC={result["roc_auc"]:.3f}')
    axes[1,1].plot([0,1], [0,1], 'k--', alpha=0.3)
    axes[1,1].set_title('ROC Curve')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    # PR curve
    precision, recall, _ = precision_recall_curve(result['labels'], result['probabilities'][:, 0], pos_label=0)
    axes[1,2].plot(recall, precision, linewidth=2)
    axes[1,2].set_title('Precision-Recall Curve')
    axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'analysis_{result["config_name"]}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"📊 Saved: analysis_{result['config_name']}.png")

In [ ]:
# Download best model
from google.colab import files

best_model_file = f"{best_result['config_name']}.pth"
print(f"📥 Downloading best model: {best_model_file}")
files.download(best_model_file)

if len(all_results) > 1:
    files.download('comprehensive_analysis.png')
else:
    files.download(f"analysis_{all_results[0]['config_name']}.png")

print("\n✅ Downloads complete!")
print(f"\n🏆 Best model: {best_result['config_name']}")
print(f"   Bot Recall: {best_result['bot_recall']:.4f}")
print(f"   Bot F1: {best_result['bot_f1']:.4f}")

## 📦 Export & Download All Results

Export comprehensive results to CSV and JSON formats, then download everything including models, visualizations, and result files.

In [ ]:
# Export results to CSV and JSON
import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create results summary DataFrame
results_data = []
for result in all_results:
    results_data.append({
        'config_name': result['config_name'],
        'model': result['model'],
        'class_weight_bot': result['class_weight_bot'],
        'test_accuracy': f"{result['test_accuracy']:.2f}%",
        'bot_recall': f"{result['bot_recall']:.4f}",
        'bot_precision': f"{result['bot_precision']:.4f}",
        'bot_f1': f"{result['bot_f1']:.4f}",
        'roc_auc': f"{result['roc_auc']:.4f}",
        'training_time_min': f"{result['training_time']/60:.1f}",
    })

results_df = pd.DataFrame(results_data)

# Export to CSV
csv_filename = f'results_summary_{timestamp}.csv'
results_df.to_csv(csv_filename, index=False)
print(f"✅ Exported summary to: {csv_filename}")

# Export detailed JSON with all metrics
json_data = {
    'timestamp': timestamp,
    'experiment_type': 'comparison' if len(all_results) > 1 else 'single',
    'num_configs': len(all_results),
    'best_model': {
        'config_name': best_result['config_name'],
        'bot_recall': float(best_result['bot_recall']),
        'bot_precision': float(best_result['bot_precision']),
        'bot_f1': float(best_result['bot_f1']),
        'test_accuracy': float(best_result['test_accuracy']),
        'roc_auc': float(best_result['roc_auc']),
    },
    'all_results': []
}

for result in all_results:
    json_data['all_results'].append({
        'config_name': result['config_name'],
        'model': result['model'],
        'class_weight_bot': result['class_weight_bot'],
        'metrics': {
            'test_accuracy': float(result['test_accuracy']),
            'bot_recall': float(result['bot_recall']),
            'bot_precision': float(result['bot_precision']),
            'bot_f1': float(result['bot_f1']),
            'roc_auc': float(result['roc_auc']),
            'training_time_min': float(result['training_time'] / 60),
        },
        'confusion_matrix': result['confusion_matrix'].tolist(),
        'training_history': {
            'epochs': list(range(1, len(result['history']['train_loss']) + 1)),
            'train_loss': [float(x) for x in result['history']['train_loss']],
            'train_acc': [float(x) for x in result['history']['train_acc']],
            'test_loss': [float(x) for x in result['history']['test_loss']],
            'test_acc': [float(x) for x in result['history']['test_acc']],
            'bot_recall': [float(x) for x in result['history']['bot_recall']],
            'bot_precision': [float(x) for x in result['history']['bot_precision']],
            'bot_f1': [float(x) for x in result['history']['bot_f1']],
        }
    })

json_filename = f'results_detailed_{timestamp}.json'
with open(json_filename, 'w') as f:
    json.dump(json_data, f, indent=2)
print(f"✅ Exported detailed results to: {json_filename}")

print(f"\n📊 Files ready for download:")
print(f"   - {csv_filename} (summary table)")
print(f"   - {json_filename} (detailed metrics & history)")

In [ ]:
# Download ALL files: models, results, visualizations
from google.colab import files
import glob
import os

print("📥 Downloading all experiment files...")
print("="*80)

download_count = 0

# 1. Download all trained models (.pth files)
print("\n1️⃣ Models (.pth):")
model_files = glob.glob('*.pth')
for model_file in model_files:
    print(f"   ⬇️  {model_file}")
    files.download(model_file)
    download_count += 1

# 2. Download all visualizations (.png files)
print("\n2️⃣ Visualizations (.png):")
viz_files = glob.glob('*.png')
for viz_file in viz_files:
    print(f"   ⬇️  {viz_file}")
    files.download(viz_file)
    download_count += 1

# 3. Download result files (.csv and .json)
print("\n3️⃣ Results (.csv, .json):")
result_files = glob.glob('results_*.csv') + glob.glob('results_*.json')
for result_file in result_files:
    print(f"   ⬇️  {result_file}")
    files.download(result_file)
    download_count += 1

print("\n" + "="*80)
print(f"✅ Downloaded {download_count} files!")
print("="*80)

print("\n📋 Summary:")
print(f"   • {len(model_files)} model(s)")
print(f"   • {len(viz_files)} visualization(s)")
print(f"   • {len(result_files)} result file(s)")

print(f"\n🏆 Best model: {best_result['config_name']}.pth")
print(f"   Bot Recall: {best_result['bot_recall']:.4f}")
print(f"   Bot F1: {best_result['bot_f1']:.4f}")
print(f"   Test Accuracy: {best_result['test_accuracy']:.2f}%")

print("\n💡 Next steps:")
print("   1. Upload the best model (.pth) to your VPS")
print("   2. Review visualizations to understand model performance")
print("   3. Check CSV/JSON files for detailed metrics")
print("   4. Use the best model for inference on new avatars")

## 💾 Optional: Backup to Google Drive

Run this cell to save all files to Google Drive for permanent storage.

In [ ]:
# Backup all files to Google Drive (optional)
from google.colab import drive
import shutil
import os
from datetime import datetime

# Mount Google Drive
print('📁 Mounting Google Drive...')
drive.mount('/content/drive')

# Create backup folder with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_folder = f'/content/drive/MyDrive/avatar_classifier_results_{timestamp}'
os.makedirs(backup_folder, exist_ok=True)

print(f'\n💾 Backing up to: {backup_folder}')
print('='*80)

backup_count = 0

# Backup models
print('\n📦 Backing up models...')
for model_file in glob.glob('*.pth'):
    shutil.copy(model_file, backup_folder)
    print(f'   ✓ {model_file}')
    backup_count += 1

# Backup visualizations
print('\n📊 Backing up visualizations...')
for viz_file in glob.glob('*.png'):
    shutil.copy(viz_file, backup_folder)
    print(f'   ✓ {viz_file}')
    backup_count += 1

# Backup results
print('\n📋 Backing up results...')
for result_file in glob.glob('results_*.csv') + glob.glob('results_*.json'):
    shutil.copy(result_file, backup_folder)
    print(f'   ✓ {result_file}')
    backup_count += 1

# Create a summary text file
summary_file = os.path.join(backup_folder, 'SUMMARY.txt')
with open(summary_file, 'w') as f:
    f.write(f"Avatar Classifier Training Results\n")
    f.write(f"{'='*80}\n\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Configs Run: {len(all_results)}\n\n")
    f.write(f"Best Model: {best_result['config_name']}\n")
    f.write(f"  - Bot Recall: {best_result['bot_recall']:.4f}\n")
    f.write(f"  - Bot Precision: {best_result['bot_precision']:.4f}\n")
    f.write(f"  - Bot F1: {best_result['bot_f1']:.4f}\n")
    f.write(f"  - Test Accuracy: {best_result['test_accuracy']:.2f}%\n")
    f.write(f"  - ROC AUC: {best_result['roc_auc']:.4f}\n\n")
    f.write(f"All Results:\n")
    f.write(f"{'-'*80}\n")
    for r in all_results:
        f.write(f"{r['config_name']}: Recall={r['bot_recall']:.4f}, F1={r['bot_f1']:.4f}, Acc={r['test_accuracy']:.2f}%\n")

print(f'\n   ✓ SUMMARY.txt')
backup_count += 1

print('\n' + '='*80)
print(f'✅ Backed up {backup_count} files to Google Drive!')
print(f'📁 Location: {backup_folder}')
print('='*80)
print('\n💡 Your results are now permanently stored in Google Drive!')